In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

import pandas as pd
import numpy as np
import os

In [2]:
input_dir = "../data/processed" 

In [3]:
# Load data
X_train = pd.read_csv(f"{input_dir}/small_X_train.csv")
y_train = pd.read_csv(f"{input_dir}/small_y_train.csv")

X_val = pd.read_csv(f"{input_dir}/small_X_val.csv")
y_val = pd.read_csv(f"{input_dir}/small_y_val.csv")

X_test = pd.read_csv(f"{input_dir}/small_X_test.csv")
y_test = pd.read_csv(f"{input_dir}/small_y_test.csv")

In [4]:
X_train = X_train.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)
X_val = X_val.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)
X_test = X_test.drop(columns = ["nameseq", "sequence"]).sort_index(axis=1)

y_train = y_train.drop(columns = "tf").to_numpy().ravel()
y_val = y_val.drop(columns = "tf").to_numpy().ravel()
y_test = y_test.drop(columns = "tf").to_numpy().ravel()

In [4]:
random_state = 23
max_iter = 1000

In [5]:
# Define candidate models
models = {
    "LR":
        (LogisticRegression(penalty = "l2",
                           solver = "lbfgs",
                           max_iter = max_iter,
                           random_state = random_state),
        {"selector__k": [5, 10, 15, "all"],
         "classifier__C": [0.05, 0.2, 1.0, 5.0]}),
    "ANN":
        (MLPClassifier(hidden_layer_sizes = (64, 32), # tapering
                      early_stopping = False, # validation on val
                      max_iter = max_iter,
                      random_state = random_state),
        {"selector__k": [5, 10, 15, "all"],
         "classifier__alpha": [1e-4, 1e-3, 1e-2, 1e-1]}),
    "SVM": # LinearSVC has no predict_proba by default
        (CalibratedClassifierCV(estimator = LinearSVC(penalty = "l2",
                                                     dual = False,
                                                     max_iter = max_iter,
                                                     random_state = random_state),
                                cv = 3,
                                ensemble = False),
        {"selector__k": [5, 10, 15, "all"],
         "classifier__estimator__C": [0.05, 0.2, 1.0, 5.0]}),
    "RF":
        (RandomForestClassifier(n_estimators = 100,
                            max_features = "sqrt",
                            n_jobs = 4, # based on machine specs
                            random_state = random_state),
        {"selector__k": [5, 10, 15, "all"],
         "classifier__max_depth": [5, 10, None],
         "classifier__min_samples_leaf": [2, 5, 10],
         "classifier__ccp_alpha": [0.0, 0.005, 0.01]})
}

In [ ]:
# set up training and validation pipeline
def train_val_test(name, model, grid):
    best_val_auc = -np.inf
    best_model = None
    best_summary = None
    
    for params in ParameterGrid(grid):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("selector", SelectKBest(score_func = f_classif)),
            ("classifier", model)
        ])
    
        pipe.set_params(**params)
        pipe.fit(X_train, y_train)
        val_auc = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1]) # positive only
    
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model = pipe
            best_summary = (name, params)
    
    test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"{best_summary[0]} | {best_summary[1]}")
    print(f"ROC-AUC: {test_auc}")
    
    y_pred = best_model.predict(X_test)
    print(classification_report(y_test,
                                y_pred,
                                labels = [True, False],
                                target_names = ["TFBS", "Non-TFBS"]
                                ))
    
    print(confusion_matrix(y_test, y_pred))

In [ ]:
# Find best Logistic Regression model
name = "LR"
model, grid = models[name]
train_val_test(name, model, grid)

In [ ]:
# Find best Artificial Neural Network model
name = "ANN"
model, grid = models[name]
train_val_test(name, model, grid)

In [ ]:
# Find best Support Vector Machine model
name = "SVM"
model, grid = models[name]
train_val_test(name, model, grid)

In [ ]:
# Find best Random Forest model
name = "ANN"
model, grid = models[name]
train_val_test(name, model, grid)